# LAB 2 — RAGAS Baseline (Naive RAG)
## Aula 5 — Avaliação de Pipelines RAG com RAGAS

Calcula as **4 métricas RAGAS** (Faithfulness, Answer Relevancy, Context Recall, Context Precision) sobre o dataset gerado no LAB 1, usando a stack:

- **LLM judge**: Groq `llama-3.1-8b-instant` (fallback Ollama `llama3.2:3b`) — wrappado em `LangchainLLMWrapper`
- **Embeddings judge**: Ollama `bge-m3` (1024d) — wrappado em `LangchainEmbeddingsWrapper`
- **Vector store**: OpenSearch 3.x (reusado do LAB 1)
- **Observabilidade**: LangFuse (opcional)

**Pré-requisito:** LAB 1 executado (`dataset_avaliacao_completo.json` + `lab1_config.json` disponíveis).

## 1. Instalação

In [ ]:
import subprocess, sys
for pkg in ['ragas>=0.1.16', 'datasets', 'pandas', 'matplotlib', 'tqdm',
            'langchain>=0.2', 'langchain-core>=0.2',
            'langchain-groq>=0.1', 'langchain-ollama>=0.1',
            'opensearch-py>=2.7', 'python-dotenv>=1.0', 'langfuse>=2.36']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✓ dependências instaladas')

## 2. Stack (Groq + Ollama + OpenSearch) + Wrappers RAGAS

In [ ]:
import os, json, warnings
from pathlib import Path
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

for env_path in [Path.cwd()/'.env', Path.home()/'mba-rag'/'.env',
                  Path.cwd().parent.parent/'.env']:
    if env_path.exists():
        load_dotenv(env_path, override=True); break

GROQ_API_KEY       = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL',   'llama-3.1-8b-instant')
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',    'http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL',   'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage
from ragas.llms       import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Selecionar LLM judge: Groq primário, Ollama fallback
lc_llm = None; LLM_PROVIDER = None; LLM_MODEL = None
if GROQ_API_KEY:
    try:
        c = ChatGroq(model=GROQ_LLM_MODEL, api_key=GROQ_API_KEY,
                     temperature=0.0, max_tokens=512, timeout=30)
        c.invoke([HumanMessage(content='ok')])
        lc_llm, LLM_PROVIDER, LLM_MODEL = c, 'groq', GROQ_LLM_MODEL
    except Exception as e:
        print(f'Groq indisponível ({e.__class__.__name__}). Fallback Ollama.')
if lc_llm is None:
    lc_llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL,
                        temperature=0.0, num_predict=512)
    LLM_PROVIDER, LLM_MODEL = 'ollama', OLLAMA_LLM_MODEL

lc_embed = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
_ = lc_embed.embed_query('teste'); assert len(_) == 1024

# Wrappers RAGAS
ragas_llm        = LangchainLLMWrapper(lc_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(lc_embed)

print(f'Stack RAGAS ✓ | LLM judge={LLM_PROVIDER}/{LLM_MODEL} | embed judge={OLLAMA_EMBED_MODEL}')

## 3. Carregar Dataset do LAB 1

In [ ]:
DATASET_PATH = 'dataset_avaliacao_completo.json'
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f'{DATASET_PATH} não encontrado. Execute o LAB 1 primeiro.')
with open(DATASET_PATH, encoding='utf-8') as f:
    dataset = json.load(f)
print(f'✓ {len(dataset)} pares carregados de {DATASET_PATH}')

## 4. Calcular as 4 Métricas RAGAS

**Faithfulness** · **Answer Relevancy** · **Context Recall** · **Context Precision** — todas usando Groq como LLM judge e Ollama BGE-M3 como embedding judge.

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision

ds = Dataset.from_list([{
    'question':     p['question'],
    'answer':       p['answer'],
    'contexts':     p['contexts'],
    'ground_truth': p['ground_truth'],
} for p in dataset])

print(f'Avaliando {len(ds)} pares com RAGAS — pode levar alguns minutos...')
result = evaluate(
    dataset    = ds,
    metrics    = [faithfulness, answer_relevancy, context_recall, context_precision],
    llm        = ragas_llm,
    embeddings = ragas_embeddings,
    raise_exceptions = False,
)

df_scores = result.to_pandas()
print('\n=== Médias RAGAS ===')
for m in ['faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']:
    print(f'  {m:22s} {df_scores[m].mean():.4f}')

## 5. Anexar Metadados e Verificar Metas do Syllabus

In [ ]:
META = {'faithfulness': 0.80, 'answer_relevancy': 0.75,
        'context_recall': 0.70, 'context_precision': 0.70}

for col in ['id', 'tipo', 'dificuldade']:
    df_scores[col] = [p.get(col, '') for p in dataset]

print('=== Status vs Meta Mínima ===')
print(f"{'métrica':<22s}{'média':>8s}{'meta':>8s}{'status':>10s}")
for m, target in META.items():
    med = df_scores[m].mean()
    st  = '✅ OK' if med >= target else '⚠ ABAIXO'
    print(f'{m:<22s}{med:8.4f}{target:8.2f}{st:>10s}')

## 6. Análise por Tipo Jurídico e Dificuldade

In [ ]:
print('\n=== Faithfulness médio por tipo jurídico ===')
print(df_scores.groupby('tipo')['faithfulness'].mean().round(3).sort_values(ascending=False))

print('\n=== Faithfulness médio por dificuldade ===')
print(df_scores.groupby('dificuldade')['faithfulness'].mean().round(3))

print('\n=== Worst-5 (Faithfulness) ===')
print(df_scores.nsmallest(5, 'faithfulness')[['id','tipo','dificuldade','faithfulness']].to_string(index=False))

## 7. Visualização — 4 Métricas vs Metas

In [ ]:
import matplotlib.pyplot as plt, numpy as np

metricas = list(META.keys())
medias   = [df_scores[m].mean() for m in metricas]
metas    = [META[m] for m in metricas]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metricas))
ax.bar(x, medias, color='#4C72B0', alpha=0.85, edgecolor='black', label='Score médio')
ax.plot(x, metas, 'r--', marker='D', label='Meta mínima', linewidth=2)
ax.set_xticks(x); ax.set_xticklabels(metricas, rotation=15)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title(f'RAGAS Baseline — Naive RAG (LLM judge: {LLM_PROVIDER}/{LLM_MODEL})',
             fontweight='bold')
ax.grid(axis='y', alpha=0.3); ax.legend()
for xi, vi in zip(x, medias):
    ax.text(xi, vi + 0.02, f'{vi:.3f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.savefig('lab2_ragas_baseline.png', dpi=140); plt.show()
print('✓ gráfico salvo: lab2_ragas_baseline.png')

## 8. Registrar no LangFuse (opcional)

In [ ]:
from langfuse import Langfuse
LANGFUSE_HOST = os.getenv('LANGFUSE_HOST',       'http://localhost:3000')
LANGFUSE_PK   = os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-xxxx')
LANGFUSE_SK   = os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-xxxx')
try:
    lf = Langfuse(public_key=LANGFUSE_PK, secret_key=LANGFUSE_SK, host=LANGFUSE_HOST)
    trace = lf.trace(
        name='Aula5-LAB2-RAGAS-Baseline-Naive-RAG',
        metadata={'aula': '5', 'lab': '2', 'n_pares': len(dataset),
                   'llm_provider': LLM_PROVIDER, 'llm_model': LLM_MODEL,
                   'embed_model': OLLAMA_EMBED_MODEL}
    )
    for m in META:
        trace.score(name=m, value=float(df_scores[m].mean()),
                    comment=f'média baseline Naive RAG ({len(dataset)} pares)')
    lf.flush()
    print(f'✓ trace LangFuse: {LANGFUSE_HOST}/traces/{trace.id}')
except Exception as e:
    print(f'LangFuse indisponível: {e}')

## 9. Exportar Resultados (entrada do LAB 5)

In [ ]:
OUT_CSV = 'ragas_baseline_resultados.csv'
df_scores.to_csv(OUT_CSV, index=False, encoding='utf-8')
print(f'✓ {OUT_CSV} ({os.path.getsize(OUT_CSV)/1024:.1f} KB)')

agg = {m: float(df_scores[m].mean()) for m in META}
agg.update({'llm_provider': LLM_PROVIDER, 'llm_model': LLM_MODEL,
             'embed_model': OLLAMA_EMBED_MODEL, 'n_pares': len(dataset)})
with open('lab2_baseline_agregado.json', 'w', encoding='utf-8') as f:
    json.dump(agg, f, ensure_ascii=False, indent=2)
print('✓ lab2_baseline_agregado.json')

## Referências (ABNT)

ES, S. et al. **RAGAS: Automated Evaluation of Retrieval Augmented Generation**. arXiv:2309.15217, 2023.

RAGAS Docs. **Customising Models — LangchainLLMWrapper / LangchainEmbeddingsWrapper**. <https://docs.ragas.io/en/stable/howtos/customisations/customize_models.html>.

GROQ INC. **Groq API Documentation**. <https://console.groq.com/docs>.

OLLAMA. **bge-m3 model**. <https://ollama.com/library/bge-m3>.

LANGFUSE. **Scores API**. <https://langfuse.com/docs/scores>.